In [1]:
from tqdm import tqdm
import urllib
import pandas as pd

In [5]:
def load_politican_edges(theme):
    with urllib.request.urlopen(f"https://raw.githubusercontent.com/Somon8/social_graphs_25/main/final-project/voting-data/politician_edges_{theme}.csv") as response:
        df = pd.read_csv(response)
    return df

big_df = load_politican_edges("klima")

In [ ]:
big_df

,source,target,weight,total_votes_agreed,total_votes_shared,source_party,target_party,period,topic
0,Frank Aaen,Alex Ahrendtsen,0.480000,24,50,Enhedslisten,Dansk Folkeparti,68,klima
1,Frank Aaen,Bjarne Corydon,0.777778,7,9,Enhedslisten,Socialdemokratiet,68,klima
2,Frank Aaen,Karsten Lauritzen,0.541667,39,72,Enhedslisten,Venstre,68,klima
3,Frank Aaen,Henrik Sass Larsen,0.694444,25,36,Enhedslisten,Socialdemokratiet,68,klima
4,Frank Aaen,Søren Espersen,0.695652,16,23,Enhedslisten,Danmarksdemokraterne,68,klima
...,...,...,...,...,...,...,...,...,...
43156,Mona Juul,Yildiz Akdogan,0.000000,0,0,Det Konservative Folkeparti,Socialdemokratiet,71,klima
43157,Mona Juul,Henrik Møller,0.000000,0,0,Det Konservative Folkeparti,Socialdemokratiet,71,klima
43158,Sjúrður Skaale,Yildiz Akdogan,0.000000,0,0,Javnaðarflokkurin,Socialdemokratiet,71,klima
43159,Sjúrður Skaale,Henrik Møller,0.000000,0,0,Javnaðarflokkurin,Socialdemokratiet,71,klima


In [39]:
def create_party_edges(topic):
    big_df = load_politican_edges(topic)

    party_edges = []
    all_periods = big_df['period'].unique()

    for period in all_periods:
        df = big_df[big_df['period'] == period]
        all_parties = list(df['source_party'].unique()) + list(df['target_party'].unique())
        unique_parties = list(set(all_parties))
        for idx, party in enumerate(unique_parties):
            for other_party in unique_parties[idx+1:]:
                #Find the sub-dataframe we should calculate it on
                mask_source = (df['source_party'] == party) & (df['target_party'] == other_party)
                mask_target = (df['source_party'] == other_party) & (df['target_party'] == party)
                mask = mask_source | mask_target
                relevant_df = df[mask]

                #Calculate the agreement
                party_votes_agreed = relevant_df["total_votes_agreed"].sum()
                party_votes_shared = relevant_df["total_votes_shared"].sum()

                party_agree_percentage = party_votes_agreed / party_votes_shared if party_votes_shared > 0 else 0

                edge = {'source': party
                    , 'target': other_party
                    , 'weight': party_agree_percentage
                    , 'total_votes_agreed': party_votes_agreed
                    , 'total_votes_shared': party_votes_shared
                    , 'period' : df['period'].iloc[0]
                    , 'topic' :df['topic'].iloc[0]
                    }
                party_edges.append(edge)

                # print(party, other_party, party_agree_percentage)
    party_df = pd.DataFrame(party_edges)
    new_df = party_df.pivot(index = ["source", "target", "topic"], columns = "period", values = ["weight"])
    new_df = new_df.reset_index()

    #Make some reasonable god damn names
    col_names = []
    for col in new_df.columns:
        if col[0] == "weight":
            name = col[1]
        else:
            name = col[0]
        col_names.append(name)
    new_df.columns = col_names



    new_df.to_csv(f"./voting-data/party_edges_{topic}.csv", index = False)
    return new_df

In [40]:
party_edges = create_party_edges(topic = "klima")